# Step 2: General Preparation (Spark Port)

## Notebook Overview

This notebook prepares the accepted-loan base table for downstream feature engineering and PD modeling in Spark.

It covers:
- loading the raw parquet exported in the previous step
- splitting the dataset into reproducible train/test subsets
- converting string-based numeric and date fields into model-ready columns
- applying train-fitted missing-value rules
- creating the binary `good_bad` target
- saving prepared parquet files for later notebooks

### Run Notes
- Keep `SAMPLE_MODE = False` when generating final artifacts.
- Any fitted statistic should be learned on `train_df` and then applied to `test_df` to avoid leakage.

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

from pyspark.sql import functions as F
from spark_utils import get_spark_session, save_parquet
from general_prep import (
    MISSING_COL,
    add_int_by_extracting_number,
    add_months_since_column,
    fit_dummy_spec,
    apply_dummy_columns,
    get_missing_summary,
    fill_null_with_column,
    fit_mean_imputer,
    apply_mean_imputer,
    fill_null_with_zero,
    check_null_count,
    add_good_bad_target,
    fit_median_imputer,
    apply_median_imputer
)

In [2]:
SAMPLE_MODE = False # True for quick development; False for full dataset

## 2A.1 Load Raw Base Data

In [3]:
spark = get_spark_session()
df = spark.read.parquet('/home/jovyan/work/data/parquet/accepted_raw.parquet')

if SAMPLE_MODE:
    df = df.sample(fraction=0.1, seed=42)

print(f'Rows: {df.count()}, Columns: {len(df.columns)}')

Spark driver memory: 7g
Rows: 2260701, Columns: 151


In [4]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double 

## 2A.2 Train/Test Split

In [5]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f'Train rows: {train_df.count()}')
print(f'Test rows: {test_df.count()}')

Train rows: 1807770
Test rows: 452931


## 2B.1 Continuous Variable Preprocessing

These transformations standardize columns that are stored as strings but behave like numeric or temporal features.

In particular, this section converts employment length and loan term into integers, then expresses key dates as month offsets from a fixed reference date so the same logic can be applied consistently to both splits.

In [6]:
train_df.select("emp_length","term","earliest_cr_line","issue_d").printSchema()

root
 |-- emp_length: string (nullable = true)
 |-- term: string (nullable = true)
 |-- earliest_cr_line: string (nullable = true)
 |-- issue_d: string (nullable = true)



In [7]:
def transform_continuous_variables(df):
    df = add_int_by_extracting_number(
        df,
        source_col="emp_length",
        target_col="emp_length_int",
        special_map={"< 1 year": 0}
    )

    df = add_int_by_extracting_number(
        df,
        source_col="term",
        target_col="term_int"
    )

    df = add_months_since_column(
        df,
        source_col='earliest_cr_line',
        output_col='mths_since_earliest_cr_line',
        reference_date='2018-12-01',
        fmt='MMM-yyyy',
    )

    df = add_months_since_column(
        df,
        source_col='issue_d',
        output_col='mths_since_issue_d',
        reference_date='2018-12-01',
        fmt='MMM-yyyy',
    )

    return df

train_df = transform_continuous_variables(train_df)
test_df  = transform_continuous_variables(test_df)

In [8]:
cols = ['emp_length', 'emp_length_int', 'term', 'term_int', 'earliest_cr_line', 'mths_since_earliest_cr_line', 'issue_d', 'mths_since_issue_d']

train_df.select(
    *cols
).show(5, truncate=False)

train_df.select(*cols).printSchema()

test_df.select(
    *cols
).show(5, truncate=False)

test_df.select(*cols).printSchema()

+----------+--------------+----------+--------+----------------+---------------------------+--------+------------------+
|emp_length|emp_length_int|term      |term_int|earliest_cr_line|mths_since_earliest_cr_line|issue_d |mths_since_issue_d|
+----------+--------------+----------+--------+----------------+---------------------------+--------+------------------+
|9 years   |9             | 36 months|36      |Jan-2004        |179                        |Aug-2017|16                |
|NULL      |NULL          | 36 months|36      |Aug-2006        |148                        |Mar-2018|9                 |
|5 years   |5             | 36 months|36      |Jun-1995        |282                        |Feb-2015|46                |
|10+ years |10            | 36 months|36      |Oct-1983        |422                        |Jan-2015|47                |
|10+ years |10            | 36 months|36      |Aug-2006        |148                        |Jan-2015|47                |
+----------+--------------+-----

In [9]:
train_df.select(
    F.min("mths_since_earliest_cr_line").alias("min"),
    F.max("mths_since_earliest_cr_line").alias("max"),
    F.avg("mths_since_earliest_cr_line").alias("mean"),
    F.expr("percentile_approx(mths_since_earliest_cr_line, 0.5)").alias("median")
).show()

+---+----+------------------+------+
|min| max|              mean|median|
+---+----+------------------+------+
| 37|1029|227.67396903509606|   212|
+---+----+------------------+------+



In [10]:
train_df.select(
    F.min("mths_since_issue_d").alias("min"),
    F.max("mths_since_issue_d").alias("max"),
    F.avg("mths_since_issue_d").alias("mean"),
    F.expr("percentile_approx(mths_since_issue_d, 0.5)").alias("median")
).show()

+---+---+-----------------+------+
|min|max|             mean|median|
+---+---+-----------------+------+
|  0|138|30.93751766705776|    29|
+---+---+-----------------+------+



## 2B.2 Discrete Variable Dummy Columns

This section is currently left commented out.

The intended workflow is to fit the dummy-column specification on `train_df` only and reuse the same specification on `test_df`, which keeps category expansions aligned and avoids train/test drift in the generated columns.

In [11]:
# discrete_cols = [
#     'grade',
#     'sub_grade',
#     'home_ownership',
#     'verification_status',
#     'purpose',
#     'addr_state',
#     'initial_list_status',
# ]

In [12]:
# dummy_spec = fit_dummy_spec(train_df, discrete_cols, max_categories=80)

# train_df = apply_dummy_columns(train_df, dummy_spec)
# test_df  = apply_dummy_columns(test_df, dummy_spec)

In [13]:
# train_dummy_cols = sorted([c for c in train_df.columns if ':' in c])
# test_dummy_cols = sorted([c for c in test_df.columns if ':' in c])

# print(len(train_dummy_cols), len(test_dummy_cols))
# print(train_dummy_cols == test_dummy_cols)

## 2B.3 Missing Value Treatment

Missing values are handled with a mix of simple, interpretable rules.

Mean and median imputers are fit on the training split, while some fields are filled from related business columns or with zero when a missing value is naturally interpreted as absence or count equals zero.

In [14]:
train_missing = get_missing_summary(train_df.select(MISSING_COL))
test_missing = get_missing_summary(test_df.select(MISSING_COL))

train_missing.show(200, truncate=False)
test_missing.show(200, truncate=False)

+---------------------------+-------------+---------------------+
|column                     |missing_count|missing_ratio        |
+---------------------------+-------------+---------------------+
|emp_length_int             |117648       |0.06507907532484775  |
|total_rev_hi_lim           |56212        |0.031094663591054172 |
|inq_last_6mths             |47           |2.599888260121586E-5 |
|mths_since_earliest_cr_line|46           |2.5445714886296378E-5|
|acc_now_delinq             |46           |2.5445714886296378E-5|
|open_acc                   |46           |2.5445714886296378E-5|
|delinq_2yrs                |46           |2.5445714886296378E-5|
|pub_rec                    |46           |2.5445714886296378E-5|
|total_acc                  |46           |2.5445714886296378E-5|
|annual_inc                 |28           |1.548869601774562E-5 |
|funded_amnt                |27           |1.4935528302826133E-5|
+---------------------------+-------------+---------------------+

+--------

In [15]:
for c in ["annual_inc", "funded_amnt"]:
    print(c)
    train_df.select(
        F.count(c).alias("count"),
        F.mean(c).alias("mean"),
        F.expr(f"percentile_approx({c}, 0.5)").alias("median"),
        F.min(c).alias("min"),
        F.max(c).alias("max"),
        F.expr(f"percentile_approx({c}, array(0.01, 0.25, 0.5, 0.75, 0.99))").alias("quantiles")
    ).show(truncate=False)

annual_inc
+-------+----------------+-------+---+-----+----------------------------------------------+
|count  |mean            |median |min|max  |quantiles                                     |
+-------+----------------+-------+---+-----+----------------------------------------------+
|1807742|77997.6486033132|65000.0|0.0|1.1E8|[16788.0, 46000.0, 65000.0, 93000.0, 270000.0]|
+-------+----------------+-------+---+-----+----------------------------------------------+

funded_amnt
+-------+------------------+-------+-----+-------+-------------------------------------------+
|count  |mean              |median |min  |max    |quantiles                                  |
+-------+------------------+-------+-----+-------+-------------------------------------------+
|1807743|15036.507180500768|12800.0|500.0|40000.0|[1550.0, 8000.0, 12800.0, 20000.0, 40000.0]|
+-------+------------------+-------+-----+-------+-------------------------------------------+



In [16]:
annual_inc_mean = fit_mean_imputer(train_df, "annual_inc")

train_df = apply_mean_imputer(train_df, "annual_inc", annual_inc_mean)
test_df  = apply_mean_imputer(test_df,  "annual_inc", annual_inc_mean)

funded_amnt_median = fit_median_imputer(train_df, "funded_amnt")

train_df = apply_median_imputer(train_df, "funded_amnt", funded_amnt_median)
test_df  = apply_median_imputer(test_df,  "funded_amnt", funded_amnt_median)

In [17]:
train_df = fill_null_with_column(train_df, "total_rev_hi_lim", "funded_amnt")
test_df  = fill_null_with_column(test_df,  "total_rev_hi_lim", "funded_amnt")

In [18]:
zero_fill_cols = [
    "mths_since_earliest_cr_line",
    "acc_now_delinq",
    "total_acc",
    "pub_rec",
    "open_acc",
    "inq_last_6mths",
    "delinq_2yrs",
    "emp_length_int",
]

train_df = fill_null_with_zero(train_df, zero_fill_cols)
test_df  = fill_null_with_zero(test_df,  zero_fill_cols)

In [19]:
check_null_count(train_df, MISSING_COL, "train_df after imputation")
check_null_count(test_df, MISSING_COL, "test_df after imputation")

=== null check: train_df after imputation ===
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+
|total_rev_hi_lim|funded_amnt|annual_inc|mths_since_earliest_cr_line|acc_now_delinq|total_acc|pub_rec|open_acc|inq_last_6mths|delinq_2yrs|emp_length_int|
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+
|0               |0          |0         |0                          |0             |0        |0      |0       |0             |0          |0             |
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+--------------+

=== null check: test_df after imputation ===
+----------------+-----------+----------+---------------------------+--------------+---------+-------+--------+--------------+-----------+

## 2B.4 Target Encoding (Good/Bad)

`good_bad` is the binary target used by later modeling notebooks.

This step collapses the original `loan_status` values into a two-class label and then checks the mapping in both train and test so class balance and status coverage are easy to verify.

In [20]:
train_df = add_good_bad_target(train_df)
test_df  = add_good_bad_target(test_df)

In [21]:
train_df.groupBy("loan_status", "good_bad") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

test_df.groupBy("loan_status", "good_bad") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

+---------------------------------------------------+--------+------+
|loan_status                                        |good_bad|count |
+---------------------------------------------------+--------+------+
|Fully Paid                                         |1       |860720|
|Current                                            |1       |703154|
|Charged Off                                        |0       |214313|
|Late (31-120 days)                                 |0       |17129 |
|In Grace Period                                    |1       |6713  |
|Late (16-30 days)                                  |1       |3476  |
|Does not meet the credit policy. Status:Fully Paid |1       |1588  |
|Does not meet the credit policy. Status:Charged Off|0       |619   |
|Default                                            |0       |31    |
|NULL                                               |1       |27    |
+---------------------------------------------------+--------+------+

+------------------

In [22]:
train_df.groupBy("good_bad").count().orderBy("good_bad").show()
test_df.groupBy("good_bad").count().orderBy("good_bad").show()

+--------+-------+
|good_bad|  count|
+--------+-------+
|       0| 232092|
|       1|1575678|
+--------+-------+

+--------+------+
|good_bad| count|
+--------+------+
|       0| 58735|
|       1|394196|
+--------+------+



## 2B.7 Save Prepared Datasets

The saved parquet files are the handoff point to the downstream feature-engineering and modeling notebooks.

At this stage, both splits should have matching schemas, the derived continuous features, the missing-value treatment applied, and the `good_bad` target column ready for reuse.

In [23]:
save_parquet(train_df, '/home/jovyan/work/data/parquet/accepted_general_prep_train.parquet')
save_parquet(test_df, '/home/jovyan/work/data/parquet/accepted_general_prep_test.parquet')

print('Prep parquet files saved.')

Saved to /home/jovyan/work/data/parquet/accepted_general_prep_train.parquet
Saved to /home/jovyan/work/data/parquet/accepted_general_prep_test.parquet
Prep parquet files saved.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 39564)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =